In [1]:
import requests
import pandas as pd
import numpy as np
import time
import json
import os
import getpass
import matplotlib.pyplot as plt
import seaborn as sns

# Setup folder
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../report", exist_ok=True)

# 1. INPUT TOKEN (Tidak tersimpan di kode)
print("Masukkan GitHub Personal Access Token Anda:")
GITHUB_TOKEN = getpass.getpass()
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
REPO = 'pandas-dev/pandas'
PANDAS_V2_DATE = pd.Timestamp("2023-04-03")
BASE_URL = f'https://api.github.com/repos/{REPO}/issues'

Masukkan GitHub Personal Access Token Anda:


In [ ]:
# 2. DATA FETCHING (Paginasi API)
def fetch_github_data(endpoint, params, max_pages=20):
    all_data = []
    print(f"Mengambil data dari {endpoint}...")
    
    for page in range(1, max_pages + 1):
        params['page'] = page
        params['per_page'] = 100
        
        response = requests.get(f"https://api.github.com/repos/{REPO}/{endpoint}", headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Berhenti di halaman {page}. Status code: {response.status_code}")
            break
            
        data = response.json()
        if not data:
            break 
            
        all_data.extend(data)
        print(f"Halaman {page} terambil ({len(all_data)} data).")
        time.sleep(1) 
        
    return all_data

# 1. Fetch Pull Requests (RQ1)
pr_params = {"state": "closed", "sort": "created", "direction": "desc"}
raw_prs = fetch_github_data("pulls", pr_params, max_pages=30) 

# 2. Fetch Issues (RQ2 & RQ3)
issue_params = {"state": "closed", "sort": "created", "direction": "desc", "since": "2020-01-01T00:00:00Z"}
raw_issues = fetch_github_data("issues", issue_params, max_pages=40)

# Simpan data mentah sebagai backup
with open("../data/raw/raw_prs.json", "w") as f: json.dump(raw_prs, f)
with open("../data/raw/raw_issues.json", "w") as f: json.dump(raw_issues, f)

In [ ]:
# 3. DATA CLEANING & TRANSFORMATION
print("\nMemulai proses pembersihan data...")
df = pd.read_csv('../data/raw/raw_dataset.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])
df['resolution_days'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / (24 * 3600)

# Hapus data dengan nilai null pada kolom krusial atau waktu resolusi negatif/0
df_clean = df.dropna(subset=['created_at', 'closed_at', 'resolution_days'])
df_clean = df_clean[df_clean['resolution_days'] > 0]

os.makedirs('../data/clean', exist_ok=True)
df_clean.to_csv('../data/clean/dataset.csv', index=False)
print(f"Data bersih disimpan! Sisa baris data valid: {len(df_clean)}")

In [ ]:
# 4. EXPLORATORY DATA ANALYSIS (EDA)
sns.set_theme(style="whitegrid")

# Plot 1: Distribusi Waktu
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean[df_clean['resolution_days'] < 60], x='resolution_days', bins=40, kde=True, color='purple')
plt.title('Distribusi Waktu Penyelesaian (Fokus < 60 Hari)')
plt.xlabel('Hari menuju Resolved')
plt.ylabel('Jumlah Issue/PR')
plt.show()

# Plot 2: Perbandingan Bug vs Enhancement
plt.figure(figsize=(7, 5))
sns.countplot(data=df_clean[df_clean['type'].isin(['bug', 'enhancement'])], x='type', palette='Set2')
plt.title('Proporsi Label: Bug vs Enhancement')
plt.show()